# Ensemble methods

Four course colabs merged: bagging/random forests on California housing, boosting on California housing,
bagging/boosting/voting on the auto MPG data, and the random forest classifier on digits.

An ensemble combines many models into one prediction. The main families:

| family | idea | sklearn |
|---|---|---|
| **bagging** | train the same model on different bootstrap samples, average the predictions. Reduces variance. | `BaggingRegressor`, `RandomForestRegressor`, `ExtraTreesRegressor` |
| **boosting** | train models one after another, each one focusing on what the previous ones got wrong. Reduces bias. | `AdaBoostRegressor`, `GradientBoostingRegressor`, `HistGradientBoostingRegressor`, (XGBoost) |
| **voting / stacking** | combine *different* kinds of models | `VotingRegressor`, `StackingRegressor` |

Every one of these has a `...Classifier` version as well.

In [1]:
# xgboost is preinstalled on Colab, uncomment if not
# %pip install -q xgboost

In [2]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, load_digits
from sklearn.ensemble import (BaggingRegressor, RandomForestRegressor, ExtraTreesRegressor,
                              AdaBoostRegressor, GradientBoostingRegressor, HistGradientBoostingRegressor,
                              VotingRegressor, StackingRegressor, RandomForestClassifier, BaggingClassifier)
from sklearn.linear_model import LinearRegression, RidgeCV
from sklearn.metrics import mean_absolute_error, r2_score
from sklearn.model_selection import (train_test_split, ShuffleSplit, KFold, cross_validate,
                                     cross_val_score, RandomizedSearchCV, GridSearchCV)
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from xgboost import XGBRegressor

np.random.seed(306)

# Part 1: California housing

Labels are multiplied by 100 so errors read in thousands of dollars.

The course colabs used 10 `ShuffleSplit` folds. Random forests and boosting on 15k rows take a while, so
this uses 5.

In [3]:
features, labels = fetch_california_housing(as_frame=True, return_X_y=True)
labels = labels * 100

train_features, test_features, train_labels, test_labels = train_test_split(
    features, labels, random_state=42)

cv = ShuffleSplit(n_splits=5, test_size=0.2, random_state=42)
summary = {}

def train_regressor(estimator, name):
    res = cross_validate(estimator, train_features, train_labels, cv=cv,
                         scoring="neg_mean_absolute_error", return_train_score=True, n_jobs=-1)
    tr, te = -res['train_score'], -res['test_score']
    summary[name] = {'train MAE': tr.mean(), 'valid MAE': te.mean(), 'valid std': te.std()}
    print(f"{name:28s} train {tr.mean():6.2f}k +/- {tr.std():.2f}k   valid {te.mean():6.2f}k +/- {te.std():.2f}k")

## Bagging

### Single tree for reference

In [4]:
train_regressor(DecisionTreeRegressor(random_state=0), "decision tree (full depth)")

decision tree (full depth)   train   0.00k +/- 0.00k   valid  47.43k +/- 1.17k


Zero training error, about 47k on validation: a fully grown tree memorises the training set.

### BaggingRegressor

10 trees (the default), each trained on a bootstrap sample (same size as the training set, drawn with
replacement, so about 63% unique rows). The predictions are averaged.

In [5]:
train_regressor(BaggingRegressor(random_state=0, n_jobs=-1), "bagging (10 trees)")

bagging (10 trees)           train  14.40k +/- 0.13k   valid  35.39k +/- 0.76k


Validation error drops from about 47k to about 35k. Each tree still overfits its own sample, but their
mistakes are different enough that they partly cancel out when averaged.

### Random forest

Bagging plus one more source of randomness: each split only considers a random subset of features
(`max_features`). That makes the trees less alike, so averaging helps more. For regression the default is
`max_features=1.0` (all features), so the extra randomness only kicks in when it's lowered.

In [6]:
train_regressor(RandomForestRegressor(n_estimators=100, random_state=0, n_jobs=-1), "random forest (100 trees)")
train_regressor(RandomForestRegressor(n_estimators=100, max_features=0.5, random_state=0, n_jobs=-1),
                "random forest, max_features=0.5")
train_regressor(ExtraTreesRegressor(n_estimators=100, max_features=0.5, random_state=0, n_jobs=-1),
                "extra trees, max_features=0.5")

random forest (100 trees)    train  12.60k +/- 0.05k   valid  33.11k +/- 0.52k


random forest, max_features=0.5 train  12.44k +/- 0.06k   valid  32.74k +/- 0.56k


extra trees, max_features=0.5 train   0.00k +/- 0.00k   valid  31.74k +/- 0.51k


More trees and a random subset of features per split each help a bit. Extra trees go further and pick
split thresholds at random too.

### Tuning a random forest

The course colab searched `n_estimators` and `max_leaf_nodes` between 2 and 100. The best result from that
search was about 40.6k, **worse** than the untuned forest above, because capping trees at 100 leaves
makes every tree underfit. Including much larger trees (and unlimited) in the search:

In [7]:
param_distributions = {
    "n_estimators": [50, 100, 200],
    "max_leaf_nodes": [100, 1000, 5000, None],
    "max_features": [0.33, 0.5, 1.0],
}
rf_search = RandomizedSearchCV(
    RandomForestRegressor(random_state=0, n_jobs=-1), param_distributions=param_distributions,
    n_iter=8, cv=3, scoring="neg_mean_absolute_error", random_state=0)
rf_search.fit(train_features, train_labels)

rf_res = pd.DataFrame(rf_search.cv_results_)
rf_res["valid MAE"] = -rf_res["mean_test_score"]
rf_res[["param_n_estimators", "param_max_leaf_nodes", "param_max_features", "valid MAE"]] \
    .sort_values("valid MAE").round(2)

,param_n_estimators,param_max_leaf_nodes,param_max_features,valid MAE
1,200,5000,0.50,33.51
4,100,None,0.50,33.67
6,100,None,0.33,34.27
0,100,5000,1.00,34.36
5,50,1000,0.50,34.48
2,100,1000,0.50,34.52
3,50,5000,1.00,34.68
7,200,100,0.33,42.05


Every configuration with big trees lands around 33.5-34.7k, which is where the untuned forest already was.
The one bad result is the forest capped at 100 leaves (about 42k), the region the course search was
limited to.

In [8]:
error = mean_absolute_error(test_labels, rf_search.predict(test_features))
print(f"tuned random forest, test MAE: {error:.2f}k")

tuned random forest, test MAE: 32.24k


Random forests usually do well with big trees. The main thing to tune is `max_features`, and more trees
only ever help (with diminishing returns), so `n_estimators` is more of a time budget than a hyperparameter.

## Boosting

### AdaBoost

Fits a sequence of trees (depth 3 by default here), increasing the weight of samples the previous trees
predicted badly.

In [9]:
train_regressor(AdaBoostRegressor(random_state=0), "AdaBoost (defaults)")

AdaBoost (defaults)          train  72.01k +/- 7.25k   valid  72.49k +/- 7.14k


AdaBoost is surprisingly bad here, around 72k, with a big spread between splits. Its default is only 50 shallow trees, and the reweighting
makes it chase the hardest samples, which in this dataset include all the blocks capped at $500k.

### Gradient boosting

Each new tree is fitted to the **residuals** (errors) of the current ensemble, and added with a small
weight (`learning_rate`). Defaults: 100 trees of depth 3.

In [10]:
train_regressor(GradientBoostingRegressor(random_state=0), "gradient boosting (defaults)")

gradient boosting (defaults) train  35.38k +/- 0.22k   valid  36.65k +/- 0.71k


### HistGradientBoosting

sklearn's faster implementation: features are binned into at most 255 values first, which makes finding
splits much cheaper. It handles missing values natively and is the one to use on anything bigger than
a few thousand rows. Same idea as LightGBM.

In [11]:
train_regressor(HistGradientBoostingRegressor(random_state=0), "hist gradient boosting")
train_regressor(HistGradientBoostingRegressor(max_iter=1000, learning_rate=0.1, early_stopping=True,
                                              random_state=0), "hist GB, 1000 iters + early stop")

hist gradient boosting       train  26.95k +/- 0.10k   valid  31.45k +/- 0.62k


hist GB, 1000 iters + early stop train  22.89k +/- 0.64k   valid  30.50k +/- 0.69k


### XGBoost

A separate library with a heavily optimised gradient boosting implementation, plus L1/L2 regularisation
on the leaf values (`reg_alpha`, `reg_lambda`).

In [12]:
train_regressor(XGBRegressor(objective='reg:squarederror', random_state=0), "XGBoost (defaults)")

XGBoost (defaults)           train  17.65k +/- 0.14k   valid  31.16k +/- 0.50k


In [13]:
xgb_big = XGBRegressor(objective='reg:squarederror', n_estimators=2000, learning_rate=0.1, max_depth=5,
                       reg_alpha=10, colsample_bytree=1, random_state=0)
train_regressor(xgb_big, "XGBoost, 2000 trees depth 5")

XGBoost, 2000 trees depth 5  train   7.13k +/- 0.09k   valid  29.65k +/- 0.46k


2000 deeper trees: validation error drops to about 30k, but training error is down around 7k. A big gap,
yet validation still improved, which is common with boosting: it can overfit the training set heavily and
still generalise better than a smaller model.

The course colab passed both `eta=0.3` and `learning_rate=0.1`. They're the same parameter under two
names, and XGBoost used `learning_rate`, which is why the two versions in the original gave identical
results.

## Comparison

In [14]:
pd.DataFrame(summary).T.round(2).sort_values("valid MAE")

,train MAE,valid MAE,valid std
"XGBoost, 2000 trees depth 5",7.13,29.65,0.46
"hist GB, 1000 iters + early stop",22.89,30.50,0.69
XGBoost (defaults),17.65,31.16,0.50
hist gradient boosting,26.95,31.45,0.62
"extra trees, max_features=0.5",0.00,31.74,0.51
"random forest, max_features=0.5",12.44,32.74,0.56
random forest (100 trees),12.60,33.11,0.52
bagging (10 trees),14.40,35.39,0.76
gradient boosting (defaults),35.38,36.65,0.71
decision tree (full depth),0.00,47.43,1.17


In [15]:
final = {
    "decision tree (depth 11)": DecisionTreeRegressor(max_depth=11, min_samples_split=5, random_state=42),
    "random forest (tuned)": rf_search.best_estimator_,
    "hist gradient boosting": HistGradientBoostingRegressor(max_iter=1000, early_stopping=True, random_state=0),
    "XGBoost, 2000 trees": xgb_big,
}
rows = {}
for name, model in final.items():
    model.fit(train_features, train_labels)
    rows[name] = mean_absolute_error(test_labels, model.predict(test_features))
pd.Series(rows, name="test MAE (k$)").round(2)

decision tree (depth 11)    42.58
random forest (tuned)       32.24
hist gradient boosting      29.42
XGBoost, 2000 trees         28.81
Name: test MAE (k$), dtype: float64

For context, on the same split: linear regression about 53k, polynomial ridge about 47k, tuned single tree
about 42k. The ensembles get down to around 30k.

---

# Part 2: Auto MPG with pipelines

Predicting fuel efficiency (miles per gallon) of 398 cars from the 1970s-80s. Smaller dataset, mixed
feature types, so the preprocessing goes in a `ColumnTransformer` and every model gets the same pipeline.

| column | type |
|---|---|
| mpg | target |
| cylinders | discrete (3-8) |
| displacement, horsepower, weight, acceleration | continuous |
| model_year | 70-82 |
| origin | 1 = USA, 2 = Europe, 3 = Japan |
| car_name | string, almost unique |

In [16]:
column_names = ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight',
                'acceleration', 'model_year', 'origin', 'car_name']
auto = pd.read_csv('https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data',
                   sep=r'\s+', header=None, names=column_names, na_values='?')
auto.shape

(398, 9)

In [17]:
auto.isna().sum()[lambda s: s > 0], auto['car_name'].nunique()

(horsepower    6
 dtype: int64,
 305)

Six cars have `?` for horsepower. Reading with `na_values='?'` turns them into `NaN` straight away. (The
original replaced `?` with -1, converted to float, and then told the imputer -1 means missing, which works
but takes three steps.) `delim_whitespace=True` is deprecated in recent pandas, `sep=r'\s+'` does the same.

`car_name` has 305 unique values for 398 cars, so it's dropped.

In [18]:
X = auto.drop(columns=['mpg', 'car_name'])
y = auto['mpg']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)

In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder

numeric_features = ['displacement', 'horsepower', 'weight', 'acceleration']
categorical_features = ['model_year', 'origin']
ordinal_features = ['cylinders']

preprocessor = ColumnTransformer([
    ("num", Pipeline([("imputer", SimpleImputer(strategy="mean")), ("scaler", StandardScaler())]), numeric_features),
    ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("ord", OrdinalEncoder(), ordinal_features),
])

def make(model):
    return Pipeline([("preprocessor", preprocessor), ("regressor", model)])

With only 318 training cars, one 80-car test set is a noisy way to compare models. Each model gets
10-fold CV R2 on the training set as well.

In [20]:
kf = KFold(n_splits=10, shuffle=True, random_state=0)

models = {
    "linear regression": LinearRegression(),
    "bagging": BaggingRegressor(random_state=0),
    "random forest": RandomForestRegressor(random_state=0),
    "gradient boosting": GradientBoostingRegressor(random_state=0),
    "AdaBoost": AdaBoostRegressor(random_state=0),
}

rows = []
for name, model in models.items():
    pipe = make(model)
    cv_r2 = cross_val_score(pipe, X_train, y_train, cv=kf, scoring="r2")
    pipe.fit(X_train, y_train)
    rows.append({"model": name, "CV R2": cv_r2.mean(), "CV std": cv_r2.std(),
                 "test R2": r2_score(y_test, pipe.predict(X_test))})

auto_results = pd.DataFrame(rows).set_index("model")
auto_results.round(3)

,CV R2,CV std,test R2
model,,,
linear regression,0.816,0.057,0.847
bagging,0.782,0.058,0.781
random forest,0.800,0.049,0.799
gradient boosting,0.820,0.050,0.813
AdaBoost,0.746,0.050,0.723


(The original printed this as `"Mean r2: {:.2f} %"`, which shows 0.78 as "0.78 %".)

### Tuning inside the pipeline

The course colab ran `GridSearchCV` on data that had already been through the preprocessor. That means the
scaler and imputer were fitted on all of the training data, including each validation fold. Searching
over the whole pipeline keeps every fold clean. Parameter names get the `regressor__` prefix.

Some of the original grids also had problems: `max_features="auto"` has been removed from random forests,
and the gradient boosting grid (`max_depth` 80-110 with up to 1000 trees, 288 combinations) was mostly
searching values that make no difference on 318 rows.

In [21]:
rf_grid = GridSearchCV(make(RandomForestRegressor(random_state=0)), {
    "regressor__n_estimators": [100, 300],
    "regressor__max_features": [0.33, 0.5, 1.0],
    "regressor__min_samples_leaf": [1, 2, 4],
}, cv=kf, scoring="r2", n_jobs=-1).fit(X_train, y_train)

gb_grid = GridSearchCV(make(GradientBoostingRegressor(random_state=0)), {
    "regressor__n_estimators": [100, 300],
    "regressor__learning_rate": [0.03, 0.1],
    "regressor__max_depth": [2, 3, 4],
    "regressor__subsample": [0.7, 1.0],
}, cv=kf, scoring="r2", n_jobs=-1).fit(X_train, y_train)

for name, g in [("random forest", rf_grid), ("gradient boosting", gb_grid)]:
    params = {k.replace("regressor__", ""): v for k, v in g.best_params_.items()}
    print(f"{name:18s} CV R2 {g.best_score_:.3f}  test R2 {g.score(X_test, y_test):.3f}  {params}")

random forest      CV R2 0.823  test R2 0.833  {'max_features': 0.33, 'min_samples_leaf': 1, 'n_estimators': 300}
gradient boosting  CV R2 0.837  test R2 0.825  {'learning_rate': 0.1, 'max_depth': 2, 'n_estimators': 300, 'subsample': 0.7}


### Voting

`VotingRegressor` averages the predictions of different kinds of models. Works best when they make
different kinds of mistakes.

In [22]:
voting = make(VotingRegressor([
    ("lr", LinearRegression()),
    ("dt", DecisionTreeRegressor(random_state=0)),
    ("svr", SVR()),
    ("knn", KNeighborsRegressor()),
]))
cv_r2 = cross_val_score(voting, X_train, y_train, cv=kf, scoring="r2")
voting.fit(X_train, y_train)
print(f"voting   CV R2 {cv_r2.mean():.3f}  test R2 {r2_score(y_test, voting.predict(X_test)):.3f}")

voting   CV R2 0.815  test R2 0.831


### Stacking

Instead of a plain average, `StackingRegressor` trains a final model on the base models' (cross-validated)
predictions, so it learns how much to trust each one.

In [23]:
stacking = make(StackingRegressor(
    estimators=[("svr", SVR(C=10)), ("knn", KNeighborsRegressor()),
                ("rf", RandomForestRegressor(random_state=0)), ("gb", GradientBoostingRegressor(random_state=0))],
    final_estimator=RidgeCV()))
cv_r2 = cross_val_score(stacking, X_train, y_train, cv=kf, scoring="r2")
stacking.fit(X_train, y_train)
print(f"stacking CV R2 {cv_r2.mean():.3f}  test R2 {r2_score(y_test, stacking.predict(X_test)):.3f}")
print("weights given to each base model:", stacking[-1].final_estimator_.coef_.round(2))

stacking CV R2 0.855  test R2 0.897
weights given to each base model: [ 0.91  0.08 -0.13  0.18]


On this small dataset plain linear regression is already hard to beat (CV R2 0.82). Default bagging and
random forest are slightly worse, tuning brings the forest and gradient boosting up to 0.82-0.84, and
stacking comes out best at about 0.855. The stacking weights show it leans mostly on the SVR, with a small
negative weight on the random forest, which mostly duplicates what gradient boosting already provides.

The test-set numbers move around a lot more than the CV ones (bagging 0.78, stacking 0.90), which is
exactly why the CV column is there.

---

# Part 3: Random forest for classification

From the "Ensemble Methods" colab, on the digits dataset.

In [24]:
X_dig, y_dig = load_digits(return_X_y=True)
Xd_train, Xd_test, yd_train, yd_test = train_test_split(X_dig, y_dig, test_size=0.2, random_state=42)
Xd_train.shape, Xd_test.shape

((1437, 64), (360, 64))

Main `RandomForestClassifier` parameters:

| parameter | |
|---|---|
| `n_estimators` | number of trees (default 100) |
| `criterion` | split quality: `gini` (default), `entropy`, `log_loss` |
| `max_depth` | maximum tree depth (default: unlimited) |
| `min_samples_split` | samples needed to split a node (default 2) |
| `min_samples_leaf` | samples needed in each leaf (default 1) |
| `max_features` | features considered per split (default `'sqrt'` for classifiers) |
| `bootstrap` | sample rows with replacement for each tree (default True) |
| `oob_score` | score each tree on the rows it didn't see, a free validation estimate |
| `class_weight` | weight classes differently, mainly for imbalanced data |
| `ccp_alpha` | cost complexity pruning, as in the decision tree notebook |

In [25]:
rfc = RandomForestClassifier(n_estimators=100, oob_score=True, random_state=0, n_jobs=-1)
rfc.fit(Xd_train, yd_train)
print(f"OOB accuracy:  {rfc.oob_score_:.3f}")
print(f"test accuracy: {rfc.score(Xd_test, yd_test):.3f}")

OOB accuracy:  0.969
test accuracy: 0.978


The out-of-bag score is close to the test score, without needing a separate validation set.

### class_weight

The course colab tried hand-picked class weights (0.2 for digit 0 up to 6 for digit 8) and got 100% test
accuracy. With 360 test images that's a difference of about 10 images and could easily be luck, and digits
is balanced anyway, so there's no reason weights should help. Checking with cross validation:

In [26]:
weights = {0: 0.2, 1: 0.4, 2: 0.6, 3: 0.8, 4: 0.8, 5: 0.9, 6: 1, 7: 3, 8: 6, 9: 1}
skf = KFold(n_splits=5, shuffle=True, random_state=0)
for name, cw in [("no class weights", None), ("hand-picked weights", weights)]:
    scores = cross_val_score(RandomForestClassifier(class_weight=cw, random_state=0, n_jobs=-1),
                             X_dig, y_dig, cv=skf)
    print(f"{name:20s} CV accuracy {scores.mean():.3f} +/- {scores.std():.3f}")

no class weights     CV accuracy 0.976 +/- 0.005


hand-picked weights  CV accuracy 0.963 +/- 0.006


If anything the hand-picked weights are slightly worse. `class_weight='balanced'` is the setting to reach for when classes are imbalanced.

### Bagging classifier

In [27]:
bag = BaggingClassifier(estimator=DecisionTreeClassifier(), n_estimators=50, oob_score=True,
                        random_state=0, n_jobs=-1)
bag.fit(Xd_train, yd_train)
print(f"single tree test accuracy: {DecisionTreeClassifier(random_state=0).fit(Xd_train, yd_train).score(Xd_test, yd_test):.3f}")
print(f"bagging test accuracy:     {bag.score(Xd_test, yd_test):.3f}   (OOB {bag.oob_score_:.3f})")
print(f"random forest accuracy:    {rfc.score(Xd_test, yd_test):.3f}")

single tree test accuracy: 0.847
bagging test accuracy:     0.942   (OOB 0.934)
random forest accuracy:    0.978


Same pattern as regression: one tree is mediocre, bagging is much better, and the random feature subsets
of a random forest add a bit more on top. (`base_estimator` was renamed to `estimator` in sklearn 1.2.)

## Summary

* **bagging / random forests** average many overfitting trees to cut variance. Little tuning needed,
  `max_features` is the main knob.
* **boosting** builds trees sequentially to cut bias. Usually the most accurate on tabular data, but more
  sensitive to settings (`learning_rate`, number of trees, depth). `HistGradientBoosting` and XGBoost are
  the practical choices.
* **voting / stacking** combine different model types. Helpful when the models are different enough.
* always tune inside a pipeline so preprocessing is refit on every fold